# Demo notebook for testing all functionality

In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../')
import numpy as np

## Set environment constraint params

In [7]:
ns_B = 8  # maxB should be on boundary (so we could always just sample the boundary...)
target_volavgB = 1.0  # tesla
ntheta_B = 16
nzeta_B = 16
len_B_field_out = ns_B * ntheta_B * nzeta_B
mirror_target = 1.35
eps_B = (mirror_target - 1.0) / (mirror_target + 1.0)
B_ub = target_volavgB * (1 + eps_B) * np.ones(len_B_field_out)  # upper bound, eq. 14
B_lb = target_volavgB * (1 - eps_B) * np.ones(len_B_field_out)  # lower bound, eq. 14

In [8]:
B_ub.shape

(2048,)

## Nonlinear inequality constraints in the acquisition function  (BoTorch)


### Background
Acquisition function optimization in Botorch (in `optimize_acqf`) uses multi-start gradient ascent from multiple initial conditions. This approach helps deal with nonconvexities and local maxima in the response surface of acquisition functions.

### Default Initialization
- Botorch normally uses a default initialization heuristic.
- This is based on batch evaluation of the acquisition function at many random points.
- It then performs Boltzmann sampling on the normalized values.
- See [`gen_batch_initial_conditions`](https://github.com/pytorch/botorch/blob/main/botorch/optim/initializers.py#L50) for implementation details.

## Challenge with Nonlinear Constraints
Sampling from the feasible set becomes very difficult with nonlinear constraints. The default BoTorch initializer doesn't support this scenario.

## Options for Handling Nonlinear Constraints

### Option 1: Provide Custom Initial Conditions
- Pass a set of initial conditions to perform multi-start ascent.
- Use the `batch_initial_conditions` argument in `optimize_acqf`.

### Option 2: Create a Custom Initializer
- Write own initializer callable with the same signature as `gen_batch_initial_conditions`.
- This should implement a strategy for generating initial conditions that satisfy the constraints.
- Pass this to `optimize_acqf` using the `ic_generator` keyword argument.

## Choosing an Approach
The best option depends on specific problem and constraints:
- If can easily generate feasible points, Option 1 might be simpler.
- For a more general or adaptive approach, Option 2 could be more suitable.

## Rejection sampling

In [21]:
x0 = np.array([0., 0.  , 2.38, 0.  , 0.  , 0.  , 2.38, 0.  ])
print(x0)
noise = 0.1*np.random.normal(0, 1, x0.shape)  # generate white noise with mean 0 and standard deviation 1
print(noise)
x0_with_noise = x0 + noise  # add the white noise to x0
print(x0_with_noise)

[0.   0.   2.38 0.   0.   0.   2.38 0.  ]
[ 0.3594844   0.14512593 -0.08213582 -0.08772212  0.19443879 -0.04571806
 -0.13614595  0.02309173]
[ 0.3594844   0.14512593  2.29786418 -0.08772212  0.19443879 -0.04571806
  2.24385405  0.02309173]


In [19]:
from src.tracing_example import get_B_field
get_B_field(np.array([0.  , 0.  , 2.38, 0.  , 0.  , 0.  , 2.38, 0.  ]))

array([0.495699  , 0.495699  , 0.495699  , ..., 0.42087505, 0.42087505,
       0.42087505])

In [15]:
# Reference: x0 = np.array([0.  , 0.  , 2.38, 0.  , 0.  , 0.  , 2.38, 0.  ])
B_lower_constraint = lambda x, cache: B_lb - get_B_field(x, cache) # eq. 14
B_upper_constraint = lambda x, cache: get_B_field(x, cache) - B_ub # eq. 14
# Number of Fourier modes for optimization
max_mode = 1
d = 4 * max_mode**2 + 4 * max_mode
lower = -1e1 # TODO: @misha - what should the lower bound be?
upper = 1e1 # TODO: @misha - what should the upper bound be?

# Sample trust region (see Turbo again) around initilisation point

def generate_random_samples_under_nonlinear_constraint(n_samples:int, d:int, upper:float, lower:float):
    samples = []
    cache = {}
    for _ in range(n_samples):
        x = np.random.uniform(upper, lower, size=(d,))
        # TODO: @misha - stop VMEC from generating crap everytime it is called
        if np.all(B_lower_constraint(x, cache) <= 0) and np.all(B_upper_constraint(x, cache) <= 0):
            samples.append(x) # Valid sample
    return np.array(samples)

In [18]:
np.random.uniform(lower, upper, size=(5, d))

array([[-9.72489266,  4.47512768, -6.37752603, -4.6253542 ,  4.50347358,
         9.45142331, -7.07050775, -6.63549647],
       [ 6.45500154,  4.59725956,  3.75044453,  7.82175539,  7.88885857,
        -1.57543484, -1.70726175, -9.13773184],
       [-0.7473909 , -7.74717647, -2.30852677,  5.04618216, -9.06135069,
         7.94027379, -9.34632524, -1.79173594],
       [ 0.98829075,  1.09554236,  8.02028844,  8.82753516,  1.28298343,
         1.84458127, -2.1260671 ,  2.05423177],
       [ 6.34622752,  8.77351642,  5.980423  , -4.50506672,  3.46934108,
         5.79653036, -4.3795666 ,  6.9421709 ]])

In [16]:
# Not vectorised approach
samples = generate_random_samples_under_nonlinear_constraint(10, d, upper, lower)